# 04 - Baseline de imagen (CNN 2.5D, un plano, un backbone)

Objetivo (plan Fase 4, ver docs/superpowers/specs/2026-08-17-fase4-baseline-cnn-design.md): tripletas 2.5D sobre el plano sagital, un solo backbone (EfficientNet-B0), entrenado solo con las 58 filas gold, validado con 3-fold CV estratificado. Sin pooling por atencion multi-plano todavia - eso es Fase 6, una vez Fase 5 anada las filas weak y haya mas datos para soportar ese modelo sin sobreajustar. Construido celda a celda: cada celda de abajo fue corrida y confirmada en Kaggle antes de anadir la siguiente.

In [1]:
# Corre en Kaggle (mas adelante en este mismo notebook hara falta GPU
# para el entrenamiento). Autocontenido, sin `from src import ...`
# (Kaggle no monta este repo, solo el notebook).
import random
from pathlib import Path

import numpy as np
import pandas as pd

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
_LOCAL_RAW = Path("../data/raw")
RAW_DIR = _KAGGLE_RAW if _KAGGLE_RAW.exists() else _LOCAL_RAW
assert RAW_DIR.exists(), f"Dataset no encontrado ni en {_KAGGLE_RAW} ni en {_LOCAL_RAW}."
print("Corriendo contra:", RAW_DIR)

OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL",
    "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus",
    "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA",
    "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA",
    "effusion": "Effusion",
    "synovitis": "Synovitis",
    "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion",
    "fracture": "Fracture",
}
FINDINGS = list(OFFICIAL_LABEL_COLUMNS.keys())
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())

random.seed(42)
np.random.seed(42)

train = pd.read_csv(RAW_DIR / "train.csv")
train_series = pd.read_csv(RAW_DIR / "train_series.csv")

n_labels_present = train[LABEL_COLS].notna().sum(axis=1)
gold_mask = n_labels_present == len(LABEL_COLS)
gold = train.loc[gold_mask].reset_index(drop=True)
print(f"gold: {len(gold)} filas")
assert len(gold) == 58, f"Se esperaban 58 filas gold, hay {len(gold)}"

gold_series = train_series[train_series["StudyInstanceUID"].isin(gold["StudyInstanceUID"])]
sag = gold_series[gold_series["Anatomical_Plane"].str.lower() == "sagittal"]
print(f"Series sagitales en gold: {len(sag)} filas, cubriendo {sag['StudyInstanceUID'].nunique()} / 58 estudios")


Corriendo contra: /kaggle/input/competitions/rsna-knee-abnormality-detection
gold: 58 filas
Series sagitales en gold: 136 filas, cubriendo 58 / 58 estudios


In [2]:
import pydicom

def count_slices(study_id, series_id):
    d = RAW_DIR / "train_series" / study_id / series_id
    return len(list(d.glob("*.dcm")))

slice_counts = {}
for _, row in sag.iterrows():
    key = row["SeriesInstanceUID"]
    slice_counts[key] = count_slices(row["StudyInstanceUID"], key)

counts = pd.Series(slice_counts)
print(f"Slices contados para {len(counts)} series sagitales")
print(counts.describe())
print(f"\nSeries con 0 cortes (carpeta vacia o no encontrada): {(counts == 0).sum()}")


Slices contados para 136 series sagitales
count    136.000000
mean      32.161765
std       30.013632
min       11.000000
25%       23.000000
50%       29.000000
75%       33.000000
max      320.000000
dtype: float64

Series con 0 cortes (carpeta vacia o no encontrada): 0


## Seleccion de una serie sagital por estudio

Regla del spec (Section 4.1): preferir `Fluid_Sensitive`, desempatar por numero de cortes, y como ultimo desempate por `SeriesInstanceUID` (lo resuelve el `sort_values` de abajo).

In [3]:
def select_sagittal_series(series_df, slice_counts):
    sagittal = series_df.copy()
    sagittal["n_slices"] = sagittal["SeriesInstanceUID"].map(slice_counts).fillna(0).astype(int)

    def _pick(group):
        fluid_sensitive = group[group["Fluid_Sensitive"] == 1]
        pool = fluid_sensitive if len(fluid_sensitive) > 0 else group
        pool = pool.sort_values(["n_slices", "SeriesInstanceUID"], ascending=[False, True])
        return pool.iloc[0]

    return sagittal.groupby("StudyInstanceUID").apply(_pick, include_groups=False)

selected = select_sagittal_series(sag, slice_counts)
print(f"Series elegidas: {len(selected)} / 58 estudios")
print(f"\nElegidas via Fluid_Sensitive: {(selected['Fluid_Sensitive'] == 1).sum()}")
print(f"Elegidas por fallback (sin Fluid_Sensitive): {(selected['Fluid_Sensitive'] == 0).sum()}")
print(f"\nDistribucion de n_slices en las series elegidas:")
print(selected["n_slices"].describe())

outliers = selected[selected["n_slices"] > 100]
print(f"\nEstudios cuya serie elegida tiene >100 cortes ({len(outliers)}):")
print(outliers[["n_slices"]] if len(outliers) else "ninguno")


Series elegidas: 58 / 58 estudios

Elegidas via Fluid_Sensitive: 56
Elegidas por fallback (sin Fluid_Sensitive): 2

Distribucion de n_slices en las series elegidas:
count     58.000000
mean      33.637931
std       38.826514
min       15.000000
25%       24.000000
50%       29.000000
75%       34.000000
max      320.000000
Name: n_slices, dtype: float64

Estudios cuya serie elegida tiene >100 cortes (1):
                                                    n_slices
StudyInstanceUID                                            
1.2.826.0.1.3680043.8.498.489465809466650318523...       320


## Investigar el outlier de 320 cortes

320 es ~10x la mediana (29) - confirmar que es una adquisicion sagital legitima (no un artefacto) antes de confiar en la regla de "mas cortes = mas completa" para este caso.

In [4]:
outlier_study = outliers.index[0]
outlier_series = selected.loc[outlier_study, "SeriesInstanceUID"]
d = RAW_DIR / "train_series" / outlier_study / outlier_series
files = sorted(d.glob("*.dcm"))
print(f"Estudio: {outlier_study}")
print(f"Serie: {outlier_series}")
print(f"Archivos .dcm: {len(files)}")

ds0 = pydicom.dcmread(files[0], stop_before_pixels=True)
print("\nSeriesDescription:", getattr(ds0, "SeriesDescription", "?"))
print("Rows x Columns:", getattr(ds0, "Rows", "?"), "x", getattr(ds0, "Columns", "?"))
print("SliceThickness:", getattr(ds0, "SliceThickness", "?"))
print("PixelSpacing:", getattr(ds0, "PixelSpacing", "?"))

slice_locs = []
for f in files:
    ds = pydicom.dcmread(f, stop_before_pixels=True)
    slice_locs.append(float(ds.SliceLocation) if "SliceLocation" in ds else np.nan)

slice_locs = pd.Series(slice_locs)
print(f"\nSliceLocation: min={slice_locs.min():.2f}, max={slice_locs.max():.2f}, "
      f"valores unicos={slice_locs.nunique()} / {len(slice_locs)}")
print(f"Rango fisico cubierto: {slice_locs.max() - slice_locs.min():.1f} mm")


Estudio: 1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132
Serie: 1.2.826.0.1.3680043.8.498.31457592019990349918394410240358497641
Archivos .dcm: 320

SeriesDescription: SAG 3D_VIEW_PD_SPAIR_HR L
Rows x Columns: 640 x 640
SliceThickness: 0.80000001192092
PixelSpacing: [0.28145694732666, 0.28145694732666]

SliceLocation: min=37.05, max=164.64, valores unicos=320 / 320
Rango fisico cubierto: 127.6 mm


**Confirmado real, no un artefacto:** secuencia 3D isotropica de alta resolucion (`SAG 3D_VIEW_PD_SPAIR_HR L`, 0.8mm de grosor, 320 SliceLocation unicos cubriendo 127.6mm). Esto expone un problema de diseno real: la densidad de cortes varia mucho entre series (11 a 320), asi que un `gap` fijo en INDICES representaria una distancia fisica muy distinta segun el estudio. Se decide (2026-08-17) medir el espaciado real por serie y definir el gap en mm, no en indices.

In [5]:
def slice_spacing_mm(study_id, series_id):
    d = RAW_DIR / "train_series" / study_id / series_id
    files = sorted(d.glob("*.dcm"))
    ds0 = pydicom.dcmread(files[0], stop_before_pixels=True)
    if "SpacingBetweenSlices" in ds0:
        return float(ds0.SpacingBetweenSlices), "SpacingBetweenSlices"
    if "SliceThickness" in ds0:
        return float(ds0.SliceThickness), "SliceThickness"
    locs = []
    for f in files:
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        if "SliceLocation" in ds:
            locs.append(float(ds.SliceLocation))
    locs.sort()
    deltas = np.diff(locs)
    return float(np.median(np.abs(deltas))), "SliceLocation deltas"

spacings = {}
sources = {}
for study_id, row in selected.iterrows():
    mm, source = slice_spacing_mm(study_id, row["SeriesInstanceUID"])
    spacings[study_id] = mm
    sources[study_id] = source

spacing_s = pd.Series(spacings)
print("Fuente del dato de espaciado:")
print(pd.Series(sources).value_counts())
print()
print("Espaciado entre cortes (mm) en las 58 series elegidas:")
print(spacing_s.describe())
print(f"\nRatio max/min: {spacing_s.max() / spacing_s.min():.2f}x")


Fuente del dato de espaciado:
SpacingBetweenSlices    52
SliceThickness           6
Name: count, dtype: int64

Espaciado entre cortes (mm) en las 58 series elegidas:
count    58.000000
mean      3.682000
std       0.818361
min       0.400000
25%       3.300000
50%       3.300000
75%       4.300000
max      5.500000
dtype: float64

Ratio max/min: 13.75x


## Gap fisico (mm) -> gap en indices, por estudio

Convierte un GAP_MM objetivo en un gap de indices distinto por estudio, usando el spacing real medido arriba. Punto de partida GAP_MM=4.0, a barrer mas adelante contra el gate de macro AUC.

In [6]:
def mm_to_slice_gap(gap_mm, spacing_mm):
    return max(1, round(gap_mm / spacing_mm))

GAP_MM = 4.0  # punto de partida a barrer mas adelante, no un valor final
gap_slices = {study_id: mm_to_slice_gap(GAP_MM, spacing_s[study_id]) for study_id in selected.index}
gap_slices_s = pd.Series(gap_slices)

print(f"gap_mm={GAP_MM} -> distribucion de gap_slices resultante por estudio:")
print(gap_slices_s.describe())
print(f"\nRango: {gap_slices_s.min()} a {gap_slices_s.max()} indices")

comparison = pd.DataFrame({
    "n_slices": selected["n_slices"],
    "spacing_mm": spacing_s,
    "gap_slices": gap_slices_s,
})
comparison["triplet_span"] = comparison["gap_slices"] * 2 + 1
comparison["margin"] = comparison["n_slices"] - comparison["triplet_span"]
print(f"\nEstudios con margen <= 2 cortes (poco espacio para el triplete):")
print(comparison[comparison["margin"] <= 2].sort_values("margin"))


gap_mm=4.0 -> distribucion de gap_slices resultante por estudio:
count    58.000000
mean      1.172414
std       1.186739
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max      10.000000
dtype: float64

Rango: 1 a 10 indices

Estudios con margen <= 2 cortes (poco espacio para el triplete):
Empty DataFrame
Columns: [n_slices, spacing_mm, gap_slices, triplet_span, margin]
Index: []


## Triplete 2.5D de extremo a extremo, un estudio de muestra

Carga los cortes reales (ordenados por SliceLocation), aplica sample_slice_indices/build_25d_triplet con el gap fisico ya calculado. Reescritas aqui como en src/features.py - si esto funciona con datos reales, esas funciones estan listas para graduarse con tests.

In [7]:
def load_series_slices(study_id, series_id):
    d = RAW_DIR / "train_series" / study_id / series_id
    files = sorted(d.glob("*.dcm"))
    records = []
    for f in files:
        ds = pydicom.dcmread(f)
        records.append((float(ds.SliceLocation), ds.pixel_array))
    records.sort(key=lambda r: r[0])
    return [pixels for _, pixels in records]

def sample_slice_indices(n_slices, n_triplets, gap):
    if n_slices <= 0 or n_triplets <= 0:
        raise ValueError("n_slices and n_triplets must be positive")
    lo, hi = gap, n_slices - 1 - gap
    if lo > hi:
        return [n_slices // 2] * n_triplets
    if n_triplets == 1:
        return [(lo + hi) // 2]
    return [int(round(x)) for x in np.linspace(lo, hi, num=n_triplets)]

def build_25d_triplet(slices, center_idx, gap):
    n = len(slices)
    idxs = [max(0, min(n - 1, center_idx + off)) for off in (-gap, 0, gap)]
    return np.stack([slices[i] for i in idxs], axis=0)

sample_study = selected.index[0]
sample_series = selected.loc[sample_study, "SeriesInstanceUID"]
slices = load_series_slices(sample_study, sample_series)
print(f"Estudio de muestra: {sample_study[:20]}...")
print(f"Cortes cargados: {len(slices)}, shape de cada corte: {slices[0].shape}, dtype: {slices[0].dtype}")

gap = gap_slices[sample_study]
center = sample_slice_indices(len(slices), n_triplets=1, gap=gap)[0]
triplet = build_25d_triplet(slices, center, gap)
print(f"\ngap_slices usado: {gap}, centro elegido: {center}")
print(f"Shape del triplete: {triplet.shape}")
print(f"Rango de valores: [{triplet.min()}, {triplet.max()}]")


Estudio de muestra: 1.2.826.0.1.3680043....
Cortes cargados: 36, shape de cada corte: (320, 320), dtype: uint16

gap_slices usado: 1, centro elegido: 17
Shape del triplete: (3, 320, 320)
Rango de valores: [0, 636]


## Normalizacion fisica y de lateralidad

normalize_physical_scale reescala por PixelSpacing real (no un tamano de pixel fijo - Fase 1); normalize_laterality voltea segun el tag DICOM Laterality. TARGET_MM_PER_PIXEL=0.35 es un punto de partida (media medida en Fase 1 entre planos), a barrer mas adelante.

In [8]:
from scipy.ndimage import zoom

def normalize_physical_scale(pixel_array, pixel_spacing_mm, target_mm_per_pixel):
    factor = pixel_spacing_mm / target_mm_per_pixel
    return zoom(pixel_array, factor, order=1)

def normalize_laterality(pixel_array, is_right_knee):
    if is_right_knee:
        return np.fliplr(pixel_array)
    return pixel_array

sample_file = sorted((RAW_DIR / "train_series" / sample_study / sample_series).glob("*.dcm"))[0]
ds0 = pydicom.dcmread(sample_file, stop_before_pixels=True)
pixel_spacing_mm = float(ds0.PixelSpacing[0])
laterality = getattr(ds0, "Laterality", None)
is_right_knee = laterality == "R"
print(f"PixelSpacing: {pixel_spacing_mm} mm/pixel, Laterality: {laterality}")

TARGET_MM_PER_PIXEL = 0.35  # punto de partida (media medida en Fase 1), a barrer mas adelante

normalized_triplet = np.stack([
    normalize_laterality(
        normalize_physical_scale(channel, pixel_spacing_mm, TARGET_MM_PER_PIXEL),
        is_right_knee,
    )
    for channel in triplet
])
print(f"Shape original: {triplet.shape} -> shape normalizado: {normalized_triplet.shape}")


PixelSpacing: 0.5 mm/pixel, Laterality: R
Shape original: (3, 320, 320) -> shape normalizado: (3, 457, 457)


## Crop fisico fijo (para poder hacer batching)

El shape normalizado varia por estudio (depende de Rows/Columns originales). No usar resize otra vez (deshace la normalizacion fisica) - crop centrado a un FOV fijo en mm, con padding si el estudio es mas pequeno. CROP_MM=130.0 viene de pilkwang/rsna-knee-baseline-v1 (ya citado en RESOURCES.md): cubre el 99.6% de las series de su corpus y conserva la articulacion.

In [9]:
def center_crop_or_pad(pixel_array, crop_px):
    h, w = pixel_array.shape
    out = np.zeros((crop_px, crop_px), dtype=pixel_array.dtype)

    src_top = max(0, (h - crop_px) // 2)
    src_left = max(0, (w - crop_px) // 2)
    src = pixel_array[src_top:src_top + crop_px, src_left:src_left + crop_px]

    dst_top = max(0, (crop_px - h) // 2)
    dst_left = max(0, (crop_px - w) // 2)
    out[dst_top:dst_top + src.shape[0], dst_left:dst_left + src.shape[1]] = src
    return out

CROP_MM = 130.0  # pilkwang/rsna-knee-baseline-v1: cubre el 99.6% de las series y conserva la articulacion
crop_px = round(CROP_MM / TARGET_MM_PER_PIXEL)
print(f"CROP_MM={CROP_MM} -> crop_px={crop_px}")

cropped_triplet = np.stack([center_crop_or_pad(channel, crop_px) for channel in normalized_triplet])
print(f"Shape final: {cropped_triplet.shape}")


CROP_MM=130.0 -> crop_px=371
Shape final: (3, 371, 371)


## Backbone real: torch + timm + EfficientNet-B0

Primera celda que usa GPU. Confirma CUDA disponible, descarga los pesos preentrenados (num_classes=0 -> features en vez de logits de ImageNet) y corre el tripete real de la celda anterior a traves de el.

In [10]:
import torch
print("torch version:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

import timm
print("timm version:", timm.__version__)

backbone = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
print(f"\nbackbone.num_features: {backbone.num_features}")

x = torch.from_numpy(cropped_triplet).unsqueeze(0).float()
print(f"Input tensor shape: {x.shape}, dtype: {x.dtype}")

backbone.eval()
with torch.no_grad():
    features = backbone(x)
print(f"Features shape: {features.shape}")


torch version: 2.10.0+cu128
CUDA disponible: True
timm version: 1.0.26

backbone.num_features: 1280
Input tensor shape: torch.Size([1, 3, 371, 371]), dtype: torch.float32
Features shape: torch.Size([1, 1280])


## Modelo completo: backbone + cabeza + learning rates diferenciales

BaselineFindingModel (backbone compartido + una cabeza lineal a los 12 hallazgos, D2L Sec. 14.2 para el fine-tuning con LR diferencial - verificado 2026-08-17, ver spec Section 5). differential_lr_param_groups materializa las listas de parametros (no deja generadores sin consumir) para que se puedan inspeccionar antes de pasarlas al optimizador sin agotarlas.

In [11]:
import torch.nn as nn

class BaselineFindingModel(nn.Module):
    def __init__(self, backbone_name, n_findings, dropout=0.5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, n_findings),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def differential_lr_param_groups(model, backbone_lr, head_lr):
    return [
        {"params": list(model.backbone.parameters()), "lr": backbone_lr},
        {"params": list(model.head.parameters()), "lr": head_lr},
    ]

model = BaselineFindingModel("efficientnet_b0", n_findings=len(FINDINGS), pretrained=True)
model = model.cuda() if torch.cuda.is_available() else model

x_gpu = x.cuda() if torch.cuda.is_available() else x
model.eval()
with torch.no_grad():
    logits = model(x_gpu)
print(f"Logits shape: {logits.shape}")  # esperado (1, 12)

param_groups = differential_lr_param_groups(model, backbone_lr=1e-5, head_lr=1e-3)
n_backbone_params = sum(p.numel() for p in param_groups[0]["params"])
n_head_params = sum(p.numel() for p in param_groups[1]["params"])
print(f"\nParametros backbone: {n_backbone_params:,} (lr={param_groups[0]['lr']})")
print(f"Parametros head: {n_head_params:,} (lr={param_groups[1]['lr']})")

optimizer = torch.optim.Adam(param_groups)
print(f"\nOptimizer creado con {len(optimizer.param_groups)} grupos")


Logits shape: torch.Size([1, 12])

Parametros backbone: 4,007,548 (lr=1e-05)
Parametros head: 15,372 (lr=0.001)

Optimizer creado con 2 grupos


## pos_weight real + prueba de humo de la loss

compute_pos_weight sobre las 58 filas gold completas, solo para probar la mecanica de principio a fin (logits del modelo -> BCEWithLogitsLoss con pos_weight). El pos_weight de entrenamiento real se recalculara por fold una vez este el split estratificado, no sobre las 58 completas (evita fuga del fold de validacion hacia el peso de la perdida).

In [12]:
def compute_pos_weight(labels):
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    safe_n_pos = n_pos.clip(lower=1)
    return torch.tensor((n_neg / safe_n_pos).to_numpy(), dtype=torch.float32)

gold_labels = gold[LABEL_COLS].reset_index(drop=True)
gold_labels.columns = FINDINGS
pos_weight = compute_pos_weight(gold_labels)
print("pos_weight por hallazgo:")
for finding, w in zip(FINDINGS, pos_weight.tolist()):
    print(f"  {finding}: {w:.2f}")

loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(logits.device))
dummy_targets = torch.zeros_like(logits)
loss = loss_fn(logits, dummy_targets)
print(f"\nLoss de prueba (contra targets todo-cero): {loss.item():.4f}")


pos_weight por hallazgo:
  acl_injury: 1.42
  mcl_injury: 5.44
  medial_meniscus_tear: 1.23
  lateral_meniscus_tear: 1.52
  oa_medial_compartment: 2.87
  oa_lateral_compartment: 4.27
  oa_patellofemoral_compartment: 1.76
  effusion: 0.66
  synovitis: 1.15
  bakers_cyst: 3.83
  bone_contusion: 2.05
  fracture: 2.22

Loss de prueba (contra targets todo-cero): 18.6901


## 3-fold CV estratificado (MultilabelStratifiedKFold)

Sechidis, Tsoumakas & Vlahavas (ECML PKDD 2011), via el paquete iterative-stratification. Sin esto, un KFold aleatorio simple no garantiza representacion estable de MCL (9 positivos en total) por fold - ver spec Section 7.

In [13]:
!pip install -q iterative-stratification


In [14]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

N_FOLDS = 3
mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

y = gold_labels[FINDINGS].to_numpy()
fold_assignment = np.zeros(len(gold_labels), dtype=int)
for fold_idx, (_, val_idx) in enumerate(mskf.split(gold_labels[FINDINGS], y)):
    fold_assignment[val_idx] = fold_idx

gold_labels["fold"] = fold_assignment
print("Estudios por fold:")
print(gold_labels["fold"].value_counts().sort_index())

print("\nPositivos por hallazgo y fold:")
per_fold_pos = gold_labels.groupby("fold")[FINDINGS].sum()
print(per_fold_pos)
print(f"\nMinimo de positivos en cualquier (fold, hallazgo): {per_fold_pos.values.min()}")


Estudios por fold:
fold
0    18
1    20
2    20
Name: count, dtype: int64

Positivos por hallazgo y fold:
(tabla completa - ver output real en Kaggle; minimo confirmado 3.0)

Minimo de positivos en cualquier (fold, hallazgo): 3.0


**Estratificacion exitosa:** MCL (el hallazgo mas raro, 9 positivos en total) queda perfectamente balanceado en 3/3/3 por fold. Minimo global entre cualquier (fold, hallazgo) es 3 - muy por encima del riesgo de 0-1 que motivo pasar de 5 a 3 folds (Section 3 del spec).

## Fix: gold_labels debe estar indexado por StudyInstanceUID

La celda de pos_weight (arriba) construyo gold_labels con `reset_index(drop=True)`, perdiendo el StudyInstanceUID como indice. No importaba para sumar columnas, pero para cruzar con `selected` (indexado por StudyInstanceUID) en el bucle de entrenamiento si importa - mejor un indice explicito que confiar en que el orden de filas coincida por casualidad.

In [15]:
gold_labels.index = gold["StudyInstanceUID"]
gold_labels.index.name = "StudyInstanceUID"

assert set(gold_labels.index) == set(selected.index), "gold_labels y selected deberian cubrir exactamente los mismos 58 estudios"
print("OK: gold_labels y selected comparten el mismo indice (StudyInstanceUID)")
print(gold_labels.head(3))


OK: gold_labels y selected comparten el mismo indice (StudyInstanceUID)
(head(3) confirmado correcto en Kaggle, incluida la columna fold)


## Preprocesar las 58 series completas

Junta todo lo validado en las celdas anteriores (seleccion, gap fisico, normalizacion, crop) en una funcion por estudio, y la corre sobre las 58 series reales (no solo la de muestra).

In [16]:
def preprocess_study(study_id):
    series_id = selected.loc[study_id, "SeriesInstanceUID"]
    slices = load_series_slices(study_id, series_id)

    d = RAW_DIR / "train_series" / study_id / series_id
    first_file = sorted(d.glob("*.dcm"))[0]
    ds0 = pydicom.dcmread(first_file, stop_before_pixels=True)
    pixel_spacing_mm = float(ds0.PixelSpacing[0])
    is_right_knee = getattr(ds0, "Laterality", None) == "R"

    gap = mm_to_slice_gap(GAP_MM, spacing_s[study_id])
    center = sample_slice_indices(len(slices), n_triplets=1, gap=gap)[0]
    raw_triplet = build_25d_triplet(slices, center, gap)

    crop_px = round(CROP_MM / TARGET_MM_PER_PIXEL)
    processed = np.stack([
        center_crop_or_pad(
            normalize_laterality(
                normalize_physical_scale(channel, pixel_spacing_mm, TARGET_MM_PER_PIXEL),
                is_right_knee,
            ),
            crop_px,
        )
        for channel in raw_triplet
    ])
    return processed.astype(np.float32)


import time
t0 = time.time()
all_processed = {}
errors = {}
for study_id in selected.index:
    try:
        all_processed[study_id] = preprocess_study(study_id)
    except Exception as e:
        errors[study_id] = repr(e)

print(f"Procesados: {len(all_processed)} / {len(selected)} en {time.time() - t0:.1f}s")
print(f"Errores: {len(errors)}")
for sid, err in errors.items():
    print(f"  {sid[:25]}...: {err}")

shapes = {sid: arr.shape for sid, arr in all_processed.items()}
print(f"\nShapes unicos: {set(shapes.values())}")


Procesados: 58 / 58 en 18.8s
Errores: 0

Shapes unicos: {(3, 371, 371)}


## Bucle de entrenamiento por fold, con weight_decay y AUC train/val por epoca

Sin augmentation todavia (decision 2026-08-17: primero ver si hace falta - ver spec Section 6). Version final de run_fold, ya con weight_decay parametrizado y reporte de train_auc en cada epoca para ver la brecha train/val en tiempo real, no solo al final. (Una primera version sin weight_decay ni train_auc corrio antes de esta - dio val macro AUC 0.4923 en el fold 0; superseded por esta version.)

In [17]:
from sklearn.metrics import roc_auc_score

def macro_roc_auc(y_true, y_pred):
    aucs = [roc_auc_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]
    return float(np.mean(aucs)), aucs


def run_fold(fold_idx, n_epochs=10, patience=3, batch_size=8, backbone_lr=1e-5, head_lr=1e-3, weight_decay=0.0):
    train_ids = gold_labels.index[gold_labels["fold"] != fold_idx]
    val_ids = gold_labels.index[gold_labels["fold"] == fold_idx]

    X_train = torch.from_numpy(np.stack([all_processed[sid] for sid in train_ids]))
    y_train = torch.from_numpy(gold_labels.loc[train_ids, FINDINGS].to_numpy().astype(np.float32))
    X_val = torch.from_numpy(np.stack([all_processed[sid] for sid in val_ids]))
    y_train_np = gold_labels.loc[train_ids, FINDINGS].to_numpy()
    y_val_np = gold_labels.loc[val_ids, FINDINGS].to_numpy()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    X_train, y_train, X_val = X_train.to(device), y_train.to(device), X_val.to(device)

    train_pos_weight = compute_pos_weight(gold_labels.loc[train_ids, FINDINGS]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=train_pos_weight)

    fold_model = BaselineFindingModel("efficientnet_b0", n_findings=len(FINDINGS), pretrained=True).to(device)
    optimizer = torch.optim.Adam(
        differential_lr_param_groups(fold_model, backbone_lr, head_lr),
        weight_decay=weight_decay,
    )

    best_val_auc = -1.0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(n_epochs):
        fold_model.train()
        perm = torch.randperm(len(X_train))
        epoch_loss = 0.0
        for start in range(0, len(perm), batch_size):
            batch_idx = perm[start:start + batch_size]
            xb, yb = X_train[batch_idx], y_train[batch_idx]
            optimizer.zero_grad()
            logits = fold_model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch_idx)
        epoch_loss /= len(X_train)

        fold_model.eval()
        with torch.no_grad():
            train_probs = torch.sigmoid(fold_model(X_train)).cpu().numpy()
            val_probs = torch.sigmoid(fold_model(X_val)).cpu().numpy()
        train_macro_auc, _ = macro_roc_auc(y_train_np, train_probs)
        val_macro_auc, _ = macro_roc_auc(y_val_np, val_probs)

        print(f"  epoch {epoch+1}/{n_epochs}: loss={epoch_loss:.4f} train_auc={train_macro_auc:.4f} val_auc={val_macro_auc:.4f}")

        if val_macro_auc > best_val_auc:
            best_val_auc = val_macro_auc
            best_state = {k: v.clone() for k, v in fold_model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"  early stopping en epoch {epoch+1}")
                break

    fold_model.load_state_dict(best_state)
    return fold_model, best_val_auc

print("=== Fold 0, weight_decay=1e-4 ===")
model_f0_wd, auc_f0_wd = run_fold(fold_idx=0, n_epochs=10, patience=3, weight_decay=1e-4)
print(f"\nMejor val macro AUC: {auc_f0_wd:.4f} (antes, sin weight_decay: 0.4923)")


=== Fold 0, weight_decay=1e-4 ===
  epoch 1/10: loss=0.9526 train_auc=0.5223 val_auc=0.5657
  epoch 2/10: loss=0.9105 train_auc=0.6812 val_auc=0.5440
  epoch 3/10: loss=0.8820 train_auc=0.8090 val_auc=0.5081
  epoch 4/10: loss=0.8528 train_auc=0.8683 val_auc=0.5274
  early stopping en epoch 4

Mejor val macro AUC: 0.5657 (antes, sin weight_decay: 0.4923)


**Diagnostico:** brecha train/val real (train sube rapido, val no acompana) - overfitting confirmado, no un modelo que simplemente no aprende. Antes de construir augmentation (la pieza mas grande que queda), probar primero si mas weight_decay solo ya alcanza.

In [18]:
print("=== Fold 0, weight_decay=1e-3 ===")
_, auc_wd1e3 = run_fold(fold_idx=0, n_epochs=10, patience=3, weight_decay=1e-3)
print(f"\nMejor val macro AUC: {auc_wd1e3:.4f}")

print("\n=== Fold 0, weight_decay=1e-2 ===")
_, auc_wd1e2 = run_fold(fold_idx=0, n_epochs=10, patience=3, weight_decay=1e-2)
print(f"\nMejor val macro AUC: {auc_wd1e2:.4f}")

print(f"\nResumen: wd=0 -> 0.4923, wd=1e-4 -> 0.5657, wd=1e-3 -> {auc_wd1e3:.4f}, wd=1e-2 -> {auc_wd1e2:.4f}")


=== Fold 0, weight_decay=1e-3 ===
  epoch 1/10: loss=0.9330 train_auc=0.5683 val_auc=0.4831
  epoch 2/10: loss=0.9256 train_auc=0.7137 val_auc=0.4770
  epoch 3/10: loss=0.8880 train_auc=0.8284 val_auc=0.4880
  epoch 4/10: loss=0.8623 train_auc=0.8809 val_auc=0.4894
  epoch 5/10: loss=0.8344 train_auc=0.9087 val_auc=0.4984
  epoch 6/10: loss=0.8179 train_auc=0.9350 val_auc=0.5019
  epoch 7/10: loss=0.7921 train_auc=0.9566 val_auc=0.5013
  epoch 8/10: loss=0.7750 train_auc=0.9607 val_auc=0.5143
  epoch 9/10: loss=0.7625 train_auc=0.9666 val_auc=0.5096
  epoch 10/10: loss=0.7419 train_auc=0.9561 val_auc=0.5102

Mejor val macro AUC: 0.5143

=== Fold 0, weight_decay=1e-2 ===
  epoch 1/10: loss=0.9494 train_auc=0.5507 val_auc=0.5774
  epoch 2/10: loss=0.9261 train_auc=0.6867 val_auc=0.5875
  epoch 3/10: loss=0.8930 train_auc=0.7927 val_auc=0.5778
  epoch 4/10: loss=0.8756 train_auc=0.8611 val_auc=0.5586
  epoch 5/10: loss=0.8547 train_auc=0.9100 val_auc=0.5586
  early stopping en epoch 5

Me

**Conclusion del barrido:** `weight_decay` ayuda algo (0.49 -> 0.59 con wd=1e-2) pero no cierra la brecha - con wd=1e-2 el train AUC sigue subiendo a 0.91 en 5 epocas mientras el val se estanca en ~0.56-0.59. weight_decay solo no alcanza; siguiente paso: augmentation (sin flip horizontal - normalize_laterality ya fija el lado medial/lateral, ver spec Section 6).

## Augmentation (sin flip horizontal)

Solo rotacion pequena, jitter de intensidad/contraste, y traslacion pequena - aplicado unicamente durante entrenamiento. **No incluye RandomHorizontalFlip**: normalize_laterality ya fija que lado es medial/lateral para 5 de los 12 hallazgos; un flip horizontal aleatorio deshace esa normalizacion y corrompe esas etiquetas para la copia volteada (D2L Sec. 14.1, verificado 2026-08-17 - ver spec Section 6). run_fold gana un parametro `augment`.

In [19]:
import torchvision.transforms as T

AUGMENTATION = T.Compose([
    T.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
])

def run_fold(fold_idx, n_epochs=10, patience=3, batch_size=8, backbone_lr=1e-5, head_lr=1e-3, weight_decay=0.0, augment=False):
    train_ids = gold_labels.index[gold_labels["fold"] != fold_idx]
    val_ids = gold_labels.index[gold_labels["fold"] == fold_idx]

    X_train = torch.from_numpy(np.stack([all_processed[sid] for sid in train_ids]))
    y_train = torch.from_numpy(gold_labels.loc[train_ids, FINDINGS].to_numpy().astype(np.float32))
    X_val = torch.from_numpy(np.stack([all_processed[sid] for sid in val_ids]))
    y_train_np = gold_labels.loc[train_ids, FINDINGS].to_numpy()
    y_val_np = gold_labels.loc[val_ids, FINDINGS].to_numpy()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    X_train, y_train, X_val = X_train.to(device), y_train.to(device), X_val.to(device)

    train_pos_weight = compute_pos_weight(gold_labels.loc[train_ids, FINDINGS]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=train_pos_weight)

    fold_model = BaselineFindingModel("efficientnet_b0", n_findings=len(FINDINGS), pretrained=True).to(device)
    optimizer = torch.optim.Adam(
        differential_lr_param_groups(fold_model, backbone_lr, head_lr),
        weight_decay=weight_decay,
    )

    best_val_auc = -1.0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(n_epochs):
        fold_model.train()
        perm = torch.randperm(len(X_train))
        epoch_loss = 0.0
        for start in range(0, len(perm), batch_size):
            batch_idx = perm[start:start + batch_size]
            xb, yb = X_train[batch_idx], y_train[batch_idx]
            if augment:
                xb = AUGMENTATION(xb)
            optimizer.zero_grad()
            logits = fold_model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch_idx)
        epoch_loss /= len(X_train)

        fold_model.eval()
        with torch.no_grad():
            train_probs = torch.sigmoid(fold_model(X_train)).cpu().numpy()
            val_probs = torch.sigmoid(fold_model(X_val)).cpu().numpy()
        train_macro_auc, _ = macro_roc_auc(y_train_np, train_probs)
        val_macro_auc, _ = macro_roc_auc(y_val_np, val_probs)

        print(f"  epoch {epoch+1}/{n_epochs}: loss={epoch_loss:.4f} train_auc={train_macro_auc:.4f} val_auc={val_macro_auc:.4f}")

        if val_macro_auc > best_val_auc:
            best_val_auc = val_macro_auc
            best_state = {k: v.clone() for k, v in fold_model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"  early stopping en epoch {epoch+1}")
                break

    fold_model.load_state_dict(best_state)
    return fold_model, best_val_auc

print("=== Fold 0, weight_decay=1e-2 + augmentation ===")
_, auc_aug = run_fold(fold_idx=0, n_epochs=15, patience=4, weight_decay=1e-2, augment=True)
print(f"\nMejor val macro AUC: {auc_aug:.4f} (mejor sin augmentation: 0.5875)")


=== Fold 0, weight_decay=1e-2 + augmentation ===
  epoch 1/15: loss=0.9421 train_auc=0.4533 val_auc=0.4897
  epoch 2/15: loss=0.9460 train_auc=0.5368 val_auc=0.4885
  epoch 3/15: loss=0.9310 train_auc=0.4941 val_auc=0.5002
  epoch 4/15: loss=0.9141 train_auc=0.4768 val_auc=0.5035
  epoch 5/15: loss=0.9256 train_auc=0.5236 val_auc=0.4870
  epoch 6/15: loss=0.9152 train_auc=0.5342 val_auc=0.5276
  epoch 7/15: loss=0.9043 train_auc=0.4723 val_auc=0.4922
  epoch 8/15: loss=0.9072 train_auc=0.5069 val_auc=0.5465
  epoch 9/15: loss=0.9093 train_auc=0.4850 val_auc=0.4690
  epoch 10/15: loss=0.9038 train_auc=0.4950 val_auc=0.5104
  epoch 11/15: loss=0.8883 train_auc=0.4806 val_auc=0.5820
  epoch 12/15: loss=0.9128 train_auc=0.4742 val_auc=0.5504
  epoch 13/15: loss=0.9068 train_auc=0.5421 val_auc=0.5143
  epoch 14/15: loss=0.9242 train_auc=0.5015 val_auc=0.5005
  epoch 15/15: loss=0.9051 train_auc=0.5380 val_auc=0.5278
  early stopping en epoch 15

Mejor val macro AUC: 0.5820 (mejor sin augmen

**No concluyente:** el augmentation si frena el sobreajuste (train_auc se queda en 0.47-0.54 en vez de dispararse a 0.91), pero el pico de val (0.582) no supera al mejor sin augmentation (0.5875) - empate, no ganador. Con el train_auc tan bajo todavia, parece que 15 epocas no alcanzan para converger a traves del ruido del augmentation, no que el augmentation no sirva. Siguiente prueba: mas epocas/paciencia antes de descartarlo.

In [20]:
print("=== Fold 0, weight_decay=1e-2 + augmentation, 40 epocas ===")
_, auc_aug_40 = run_fold(fold_idx=0, n_epochs=40, patience=8, weight_decay=1e-2, augment=True)
print(f"\nMejor val macro AUC: {auc_aug_40:.4f}")
print(f"Comparacion: sin augmentation=0.5875, augmentation 15 epocas=0.5820, augmentation 40 epocas={auc_aug_40:.4f}")


=== Fold 0, weight_decay=1e-2 + augmentation, 40 epocas ===
  epoch 1/40: loss=0.9434 train_auc=0.5079 val_auc=0.5764
  epoch 2/40: loss=0.9312 train_auc=0.5391 val_auc=0.4462
  epoch 3/40: loss=0.9218 train_auc=0.5498 val_auc=0.5290
  epoch 4/40: loss=0.9142 train_auc=0.4979 val_auc=0.4952
  epoch 5/40: loss=0.9146 train_auc=0.5311 val_auc=0.4732
  epoch 6/40: loss=0.9246 train_auc=0.5103 val_auc=0.5147
  epoch 7/40: loss=0.9122 train_auc=0.4962 val_auc=0.5014
  epoch 8/40: loss=0.9207 train_auc=0.4888 val_auc=0.4876
  epoch 9/40: loss=0.8990 train_auc=0.5362 val_auc=0.4841
  early stopping en epoch 9

Mejor val macro AUC: 0.5764
Comparacion: sin augmentation=0.5875, augmentation 15 epocas=0.5820, augmentation 40 epocas=0.5764


Mas epocas/paciencia no ayudo (para en epoch 9, mejor resultado en epoch 1) - no es un problema de tiempo de convergencia. Probar aislar el augmentation solo, sin weight_decay=1e-2 compitiendo por el mismo trabajo de regularizar.

In [21]:
print("=== Fold 0, sin weight_decay + augmentation, 40 epocas ===")
_, auc_aug_nowd = run_fold(fold_idx=0, n_epochs=40, patience=8, weight_decay=0.0, augment=True)
print(f"\nMejor val macro AUC: {auc_aug_nowd:.4f}")
print(f"Comparacion: sin nada=0.4923, solo wd=1e-2=0.5875, wd=1e-2+aug=0.582, solo aug={auc_aug_nowd:.4f}")


=== Fold 0, sin weight_decay + augmentation, 40 epocas ===
  epoch 1/40: loss=0.9473 train_auc=0.5195 val_auc=0.5170
  epoch 2/40: loss=0.9399 train_auc=0.4888 val_auc=0.4881
  epoch 3/40: loss=0.9266 train_auc=0.4920 val_auc=0.5055
  epoch 4/40: loss=0.9265 train_auc=0.4904 val_auc=0.4635
  epoch 5/40: loss=0.9327 train_auc=0.5222 val_auc=0.4894
  epoch 6/40: loss=0.9200 train_auc=0.4785 val_auc=0.4568
  epoch 7/40: loss=0.8999 train_auc=0.5377 val_auc=0.5348
  epoch 8/40: loss=0.9007 train_auc=0.5389 val_auc=0.4390
  epoch 9/40: loss=0.9122 train_auc=0.5039 val_auc=0.4719
  epoch 10/40: loss=0.8927 train_auc=0.4691 val_auc=0.5325
  epoch 11/40: loss=0.8955 train_auc=0.4908 val_auc=0.5131
  epoch 12/40: loss=0.8963 train_auc=0.4822 val_auc=0.4627
  epoch 13/40: loss=0.8951 train_auc=0.5337 val_auc=0.4995
  epoch 14/40: loss=0.8809 train_auc=0.4656 val_auc=0.5102
  epoch 15/40: loss=0.8756 train_auc=0.5525 val_auc=0.4899
  early stopping en epoch 15

Mejor val macro AUC: 0.5348
Compara

## Resultado del barrido de regularizacion (fold 0)

| Config | Val macro AUC |
|---|---|
| Sin regularizacion | 0.4923 |
| wd=1e-4 | 0.5657 |
| wd=1e-3 | 0.5143 |
| **wd=1e-2 (mejor)** | **0.5875** |
| wd=1e-2 + augmentation | 0.5820-0.5764 |
| Solo augmentation (wd=0) | 0.5348 |

**Decision (2026-08-17):** `weight_decay=1e-2` sin augmentation es la mejor configuracion encontrada. El augmentation ayuda un poco sobre no tener nada (0.49 -> 0.53) pero es mas debil que weight_decay solo, y combinarlos no suma - resultado negativo honesto, no descartado para siempre (candidato a revisar en Fase 5 con mas datos). Siguiente paso: correr esta configuracion en los 3 folds para el numero real de macro AUC de la Fase 4.

## Resultado final: 3-fold CV, weight_decay=1e-2, sin augmentation

Configuracion ganadora del barrido anterior. Responde al gate real de la Fase 4 (spec Section 7): media entre folds, no un solo numero.

In [22]:
fold_results = {}
for fold_idx in range(N_FOLDS):
    print(f"=== Fold {fold_idx} ===")
    _, best_auc = run_fold(fold_idx=fold_idx, n_epochs=40, patience=8, weight_decay=1e-2, augment=False)
    fold_results[fold_idx] = best_auc
    print(f"Fold {fold_idx} mejor val macro AUC: {best_auc:.4f}\n")

fold_aucs = pd.Series(fold_results)
print("=== Resumen final ===")
print(fold_aucs)
print(f"\nMedia across folds: {fold_aucs.mean():.4f}")
print(f"Desviacion estandar: {fold_aucs.std():.4f}")
print(f"Baseline constante 0.5: {'GANA' if fold_aucs.mean() > 0.5 else 'NO GANA'}")


=== Fold 0 ===
  (13 epocas, early stopping - ver detalle completo en la conversacion 2026-08-17)
Fold 0 mejor val macro AUC: 0.5953

=== Fold 1 ===
  (9 epocas, early stopping)
Fold 1 mejor val macro AUC: 0.5145

=== Fold 2 ===
  (28 epocas, early stopping - train_auc llega a 0.999, memorizacion casi total)
Fold 2 mejor val macro AUC: 0.6122

=== Resumen final ===
0    0.595282
1    0.514540
2    0.612231
dtype: float64

Media across folds: 0.5740
Desviacion estandar: 0.0522
Baseline constante 0.5: GANA


## Conclusion de la Fase 4 (primer baseline, 2026-08-17)

**Resultado:** macro ROC-AUC 0.574 (std 0.052) en 3-fold CV estratificado, contra un baseline constante de 0.5 - gana, los 3 folds individualmente por encima de 0.5 (0.595, 0.515, 0.612).

**Honestidad sobre la calidad de esta victoria:** en los 3 folds el train_auc sube a 0.95-0.99 (fold 2 llega a 0.999) - memorizacion casi total del training set con solo ~38-40 estudios por fold. Que aun asi generalice a 0.574 de media es una senal real pero fragil, no un modelo robusto. El fold 1 (0.515) esta practicamente en el limite del azar.

**Configuracion ganadora:** EfficientNet-B0 (un plano sagital), gap fisico GAP_MM=4.0mm, TARGET_MM_PER_PIXEL=0.35, CROP_MM=130.0 (pilkwang), learning rate diferencial (backbone=1e-5, head=1e-3), dropout=0.5, weight_decay=1e-2, early stopping (patience=8) sobre val macro AUC, SIN augmentation.

**Augmentation: probado, no gano** (ver barrido arriba) - resultado negativo honesto, no descartado para siempre. Candidato a revisar en Fase 5 con mas datos (gold+weak), donde el overfitting extremo visto aqui deberia ser menos severo.

**Pendiente antes de tocar src/:** revision del usuario de todo este notebook (build cell-by-cell) antes de graduar cualquier pieza a src/data.py, src/features.py, src/model.py.